# Step 5. OpenCV + Silero VAD 검증 (자취방 + 요리)

**목표:** SampleVideo.mp4에서 찾은 최적 파라미터가 다른 영상에서도 일반화되는지 검증

| 모델 | 파라미터 | 최적값 (SampleVideo 기준) |
|------|----------|-------------------------|
| OpenCV | threshold | 30 |
| OpenCV | frame_interval | 5 |
| Silero VAD | threshold | 0.5 |
| Silero VAD | min_silence_ms | 1000 |

> **주의:** 위 최적값은 tune_ad_timing_colab.ipynb 실행 결과에 따라 수정하세요.

**검증 대상:**
- `SampleVideo.mp4` (9분44초 드라마) — 기준선 재확인
- `자취방.mp4` (자취방 유튜브)
- `요리.mp4` (요리 유튜브)

## 0. 환경 설정

In [ ]:
# GPU 확인
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print("GPU 확인 완료!")

In [ ]:
# 패키지 설치
!pip install -q torch torchaudio
!apt-get install -y fonts-nanum > /dev/null 2>&1
print("패키지 설치 완료!")

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

BASE = "/content/drive/MyDrive/SampleVideo.Test"

VIDEOS = {
    "SampleVideo": os.path.join(BASE, "SampleVideo.mp4"),
    "자취방":       os.path.join(BASE, "자취방.mp4"),
    "요리":         os.path.join(BASE, "요리.mp4"),
}

for name, path in VIDEOS.items():
    exists = os.path.exists(path)
    size = f"{os.path.getsize(path)/1024/1024:.1f}MB" if exists else "없음"
    print(f"[{name}] {size} {'OK' if exists else 'MISSING'}")

print("\n경로 확인 완료!")

## 1. 최적 파라미터 설정 (SampleVideo 튜닝 결과)

In [ ]:
# ============================================================
#  SampleVideo.mp4 튜닝에서 확정된 최적 파라미터
#  tune_ad_timing_colab.ipynb 결과에 맞춰 수정하세요!
# ============================================================

# OpenCV 모션 탐지
BEST_OPENCV_THRESHOLD = 30
BEST_OPENCV_INTERVAL  = 5

# Silero VAD 침묵 탐지
BEST_SILERO_THRESHOLD = 0.5
BEST_SILERO_MIN_MS    = 1000

print("=" * 50)
print("  검증용 최적 파라미터")
print("=" * 50)
print(f"  [OpenCV]     threshold={BEST_OPENCV_THRESHOLD}, frame_interval={BEST_OPENCV_INTERVAL}")
print(f"  [Silero VAD] threshold={BEST_SILERO_THRESHOLD}, min_silence_ms={BEST_SILERO_MIN_MS}")
print("=" * 50)

## 2. 분석 함수 정의

In [ ]:
import cv2
import torch
import torchaudio
import numpy as np
import subprocess
import time
import tempfile

# ── Silero VAD 모델 로드
silero_model, silero_utils = torch.hub.load(
    repo_or_dir='snakers4/silero-vad',
    model='silero_vad',
    force_reload=False
)
(get_speech_timestamps, _, read_audio, _, _) = silero_utils
print("Silero VAD 모델 로드 완료!")


def analyze_opencv(video_path, threshold, frame_interval):
    """OpenCV 모션 탐지 — 저모션 구간 반환"""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps

    prev_gray = None
    motion_scores = []
    frame_indices = []
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % frame_interval == 0:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            gray = cv2.GaussianBlur(gray, (21, 21), 0)
            if prev_gray is not None:
                diff = cv2.absdiff(prev_gray, gray)
                _, thresh_img = cv2.threshold(diff, threshold, 255, cv2.THRESH_BINARY)
                score = cv2.countNonZero(thresh_img)
                motion_scores.append(score)
                frame_indices.append(frame_idx)
            prev_gray = gray
        frame_idx += 1
    cap.release()

    # 저모션 구간: 모션 점수가 전체 중앙값 이하
    if len(motion_scores) == 0:
        return {'duration': duration, 'fps': fps, 'low_motion_count': 0,
                'total_analyzed': 0, 'low_motion_ratio': 0,
                'avg_motion': 0, 'low_motion_intervals': []}

    median_score = np.median(motion_scores)
    low_motion_indices = [frame_indices[i] for i, s in enumerate(motion_scores) if s <= median_score * 0.3]

    # 연속 저모션 프레임을 구간으로 그룹화
    intervals = []
    if low_motion_indices:
        start = low_motion_indices[0]
        prev = start
        for idx in low_motion_indices[1:]:
            if idx - prev > frame_interval * 2:
                intervals.append((start / fps, prev / fps))
                start = idx
            prev = idx
        intervals.append((start / fps, prev / fps))

    # 최소 0.5초 이상 구간만
    intervals = [(s, e) for s, e in intervals if e - s >= 0.5]

    return {
        'duration': duration,
        'fps': fps,
        'total_analyzed': len(motion_scores),
        'low_motion_count': len(low_motion_indices),
        'low_motion_ratio': len(low_motion_indices) / len(motion_scores) * 100,
        'avg_motion': np.mean(motion_scores),
        'low_motion_intervals': intervals
    }


def analyze_silero(video_path, threshold, min_silence_ms):
    """Silero VAD 침묵 탐지 — 침묵 구간 반환"""
    # 오디오 추출
    tmp_wav = tempfile.mktemp(suffix='.wav')
    subprocess.run(['ffmpeg', '-y', '-i', video_path, '-ar', '16000',
                    '-ac', '1', '-f', 'wav', tmp_wav],
                   capture_output=True)

    wav = read_audio(tmp_wav, sampling_rate=16000)
    duration = len(wav) / 16000

    # 음성 구간 탐지
    speech_timestamps = get_speech_timestamps(
        wav, silero_model,
        threshold=threshold,
        min_silence_duration_ms=min_silence_ms,
        sampling_rate=16000
    )

    # 음성 구간의 여집합 = 침묵 구간
    silence_intervals = []
    prev_end = 0
    for ts in speech_timestamps:
        start_s = ts['start'] / 16000
        end_s = ts['end'] / 16000
        if start_s - prev_end > 0.1:
            silence_intervals.append((prev_end, start_s))
        prev_end = end_s
    if duration - prev_end > 0.1:
        silence_intervals.append((prev_end, duration))

    total_silence = sum(e - s for s, e in silence_intervals)

    os.remove(tmp_wav)

    return {
        'duration': duration,
        'silence_count': len(silence_intervals),
        'total_silence': total_silence,
        'silence_ratio': total_silence / duration * 100,
        'avg_silence_len': total_silence / len(silence_intervals) if silence_intervals else 0,
        'silence_intervals': silence_intervals
    }


def find_ad_timings(low_motion_intervals, silence_intervals, min_overlap=0.5):
    """모션 낮은 구간 + 침묵 구간 교집합 → 광고 타이밍"""
    ad_timings = []
    for ms, me in low_motion_intervals:
        for ss, se in silence_intervals:
            overlap_start = max(ms, ss)
            overlap_end = min(me, se)
            if overlap_end - overlap_start >= min_overlap:
                ad_timings.append((overlap_start, overlap_end))
    # 중복 제거 및 정렬
    ad_timings.sort()
    merged = []
    for s, e in ad_timings:
        if merged and s <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], e))
        else:
            merged.append((s, e))
    return merged

print("분석 함수 정의 완료!")

## 3. 3개 영상 분석 실행

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

results = {}

for name, path in tqdm(VIDEOS.items(), desc="영상 분석"):
    print(f"\n{'='*55}")
    print(f"  [{name}] 분석 시작")
    print(f"{'='*55}")

    # OpenCV 모션 탐지
    t0 = time.time()
    res_cv = analyze_opencv(path, BEST_OPENCV_THRESHOLD, BEST_OPENCV_INTERVAL)
    cv_time = time.time() - t0
    print(f"  OpenCV: 저모션 구간 {len(res_cv['low_motion_intervals'])}개 ({cv_time:.1f}s)")

    # Silero VAD 침묵 탐지
    t0 = time.time()
    res_sil = analyze_silero(path, BEST_SILERO_THRESHOLD, BEST_SILERO_MIN_MS)
    sil_time = time.time() - t0
    print(f"  Silero: 침묵 구간 {res_sil['silence_count']}개, 침묵 비율 {res_sil['silence_ratio']:.1f}% ({sil_time:.1f}s)")

    # 광고 타이밍 (교집합)
    ad_timings = find_ad_timings(res_cv['low_motion_intervals'], res_sil['silence_intervals'])
    print(f"  광고 후보 타이밍: {len(ad_timings)}개")

    for i, (s, e) in enumerate(ad_timings[:5]):
        print(f"    [{i+1}] {s:.1f}s ~ {e:.1f}s (길이: {e-s:.1f}s)")
    if len(ad_timings) > 5:
        print(f"    ... 외 {len(ad_timings)-5}개")

    results[name] = {
        'duration': res_cv['duration'],
        'opencv': res_cv,
        'silero': res_sil,
        'ad_timings': ad_timings,
        'cv_time': cv_time,
        'sil_time': sil_time
    }

print("\n모든 영상 분석 완료!")

## 4. 결과 비교 테이블

In [ ]:
from IPython.display import display, HTML

# ── 요약 테이블
summary_rows = []
for name, res in results.items():
    summary_rows.append({
        '영상': name,
        '영상 길이(s)': f"{res['duration']:.1f}",
        '저모션 구간 수': len(res['opencv']['low_motion_intervals']),
        '침묵 구간 수': res['silero']['silence_count'],
        '침묵 비율(%)': f"{res['silero']['silence_ratio']:.1f}",
        '평균 침묵 길이(s)': f"{res['silero']['avg_silence_len']:.1f}",
        '광고 후보 수': len(res['ad_timings']),
        '처리 시간(s)': f"{res['cv_time'] + res['sil_time']:.1f}"
    })

df_summary = pd.DataFrame(summary_rows)
print("=" * 60)
print("  Step 5. 3개 영상 검증 결과 요약")
print(f"  OpenCV: threshold={BEST_OPENCV_THRESHOLD}, frame_interval={BEST_OPENCV_INTERVAL}")
print(f"  Silero: threshold={BEST_SILERO_THRESHOLD}, min_silence_ms={BEST_SILERO_MIN_MS}")
print("=" * 60)
display(df_summary)

# ── 광고 타이밍 상세
print("\n" + "=" * 60)
print("  광고 후보 타이밍 상세")
print("=" * 60)
ad_rows = []
for name, res in results.items():
    for i, (s, e) in enumerate(res['ad_timings']):
        mins, secs = divmod(s, 60)
        ad_rows.append({
            '영상': name,
            '번호': i+1,
            '시작(s)': f"{s:.1f}",
            '종료(s)': f"{e:.1f}",
            '길이(s)': f"{e-s:.1f}",
            '시작 시각': f"{int(mins):02d}:{secs:04.1f}"
        })
df_ad = pd.DataFrame(ad_rows)
display(df_ad)

## 5. 시각화

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib

# 한글 폰트 설정
fm._load_fontmanager(try_read_cache=False)
nanum_fonts = [f.fname for f in fm.fontManager.ttflist if 'Nanum' in f.name]
if nanum_fonts:
    font_prop = fm.FontProperties(fname=nanum_fonts[0])
    matplotlib.rcParams['font.family'] = font_prop.get_name()
else:
    font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    fm.fontManager.addfont(font_path)
    font_prop = fm.FontProperties(fname=font_path)
    matplotlib.rcParams['font.family'] = font_prop.get_name()
matplotlib.rcParams['axes.unicode_minus'] = False

video_names = list(results.keys())
n_videos = len(video_names)

fig, axes = plt.subplots(n_videos, 1, figsize=(18, 4 * n_videos))
fig.suptitle(f'OpenCV + Silero VAD  Ad Timing Detection\n'
             f'(threshold={BEST_OPENCV_THRESHOLD}, interval={BEST_OPENCV_INTERVAL}, '
             f'silero_th={BEST_SILERO_THRESHOLD}, min_ms={BEST_SILERO_MIN_MS})',
             fontsize=13, fontweight='bold')

for i, name in enumerate(video_names):
    ax = axes[i] if n_videos > 1 else axes
    res = results[name]
    dur = res['duration']

    # 타임라인 바
    ax.barh(0, dur, height=0.3, color='lightgray', label='전체 영상')

    # 저모션 구간
    for s, e in res['opencv']['low_motion_intervals']:
        ax.barh(0, e-s, left=s, height=0.3, color='skyblue', alpha=0.6)

    # 침묵 구간
    for s, e in res['silero']['silence_intervals']:
        ax.barh(1, e-s, left=s, height=0.3, color='salmon', alpha=0.6)

    # 광고 타이밍 (교집합)
    for s, e in res['ad_timings']:
        ax.barh(2, e-s, left=s, height=0.3, color='limegreen', alpha=0.8)

    ax.set_yticks([0, 1, 2])
    ax.set_yticklabels(['저모션', '침묵', '광고 타이밍'], fontproperties=font_prop, fontsize=11)
    ax.set_xlim(0, dur)
    ax.set_xlabel('시간 (초)', fontproperties=font_prop, fontsize=10)
    ax.set_title(f'{name} ({dur:.0f}초) — 광고 후보 {len(res["ad_timings"])}개',
                 fontproperties=font_prop, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('validate_ad_timing_result.png', dpi=150, bbox_inches='tight')
plt.show()
print("그래프 저장: validate_ad_timing_result.png")

## 6. 결과 저장 & 다운로드

In [ ]:
import shutil

drive_save = "/content/drive/MyDrive/SampleVideo.Test"

# CSV 저장
df_summary.to_csv('validate_ad_timing_summary.csv', index=False, encoding='utf-8-sig')
df_ad.to_csv('validate_ad_timing_detail.csv', index=False, encoding='utf-8-sig')

# Drive 백업
for f in ['validate_ad_timing_summary.csv', 'validate_ad_timing_detail.csv', 'validate_ad_timing_result.png']:
    shutil.copy(f, os.path.join(drive_save, f))
    print(f"Drive 저장: {f}")

print("\n모든 결과 저장 완료!")

## 7. 리포트용 마크다운 출력

In [ ]:
print("=" * 60)
print("  아래 내용을 Project_Report.md Step 5에 붙여넣으세요")
print("=" * 60)

print(f"\n#### Step 5. OpenCV + Silero VAD — 다른 영상 2종 검증")
print(f"\n확정 파라미터 (SampleVideo 기준 튜닝 결과):")
print(f"- OpenCV: threshold={BEST_OPENCV_THRESHOLD}, frame_interval={BEST_OPENCV_INTERVAL}")
print(f"- Silero VAD: threshold={BEST_SILERO_THRESHOLD}, min_silence_ms={BEST_SILERO_MIN_MS}")
print()
print("| 영상 | 영상 길이 | 저모션 구간 | 침묵 구간 | 침묵 비율 | 광고 후보 |")
print("|------|----------|-----------|----------|----------|----------|")
for name, res in results.items():
    dur = f"{res['duration']:.0f}s"
    lm = len(res['opencv']['low_motion_intervals'])
    sc = res['silero']['silence_count']
    sr = f"{res['silero']['silence_ratio']:.1f}%"
    ad = len(res['ad_timings'])
    print(f"| {name} | {dur} | {lm}개 | {sc}개 | {sr} | {ad}개 |")

print("\n**광고 후보 타이밍 상세:**")
print()
for name, res in results.items():
    if res['ad_timings']:
        print(f"**{name}:**")
        for i, (s, e) in enumerate(res['ad_timings']):
            mins, secs = divmod(s, 60)
            print(f"- [{i+1}] {int(mins):02d}:{secs:04.1f} ~ {e:.1f}s (길이: {e-s:.1f}s)")
        print()